# EEGMMIDB quick reader

This notebook loads a local file from the PhysioNet EEG Motor Movement/Imagery dataset (`eegmmidb`).

- File format: EDF+ (`.edf`), 64 EEG channels, 160 Hz
- Annotation codes in each run: `T0` (rest), `T1`, `T2`
- `T1`/`T2` meanings depend on run number, so we decode them using the run-task map.

In [46]:
from pathlib import Path
from collections import Counter

import mne

from mne.datasets import eegbci

# Choose a file to load
SUBJECT = 1
RUN = 6  # Try 4, 6, 10, 14 for motor imagery runs

# Candidate roots for your local download
CANDIDATE_ROOTS = [
    Path("MNE-eegbci-data/files/eegmmidb/1.0.0"),
    Path("MNE-eegbci-data/eegmmidb/1.0.0"),
    Path("files/eegmmidb/1.0.0"),
]

def resolve_data_root(candidates):
    for candidate in candidates:
        if candidate.exists():
            return candidate.resolve()
    checked = "\n".join(f"- {c.resolve()}" for c in candidates)
    raise FileNotFoundError(
        "Could not find eegmmidb root. Checked:\n" + checked
    )

data_root = resolve_data_root(CANDIDATE_ROOTS)

edf_path = data_root / f"S{SUBJECT:03d}" / f"S{SUBJECT:03d}R{RUN:02d}.edf"

if not edf_path.exists():
    raise FileNotFoundError(f"Missing file: {edf_path}")

print("Data root:", data_root)
print("EDF file:", edf_path)


Data root: C:\Users\Kades\Documents\MAU\AML GP\BCI\MNE-eegbci-data\files\eegmmidb\1.0.0
EDF file: C:\Users\Kades\Documents\MAU\AML GP\BCI\MNE-eegbci-data\files\eegmmidb\1.0.0\S001\S001R06.edf


In [47]:
raw = mne.io.read_raw_edf(edf_path, infer_types=True, preload=False, verbose="ERROR")

eegbci.standardize(raw)

raw.set_montage(mne.channels.make_standard_montage("standard_1005"), on_missing="ignore")

print(raw)
print(f"Sampling frequency: {raw.info['sfreq']} Hz")
print(f"Number of channels: {len(raw.ch_names)}")
print("First 8 channels:", raw.ch_names[:8])


<RawEDF | S001R06.edf, 64 x 20000 (125.0 s), ~75 KiB, data not loaded>
Sampling frequency: 160.0 Hz
Number of channels: 64
First 8 channels: ['FC5', 'FC3', 'FC1', 'FCz', 'FC2', 'FC4', 'FC6', 'C5']


In [48]:
events, event_id = mne.events_from_annotations(raw, verbose="ERROR")

id_to_code = {v: k for k, v in event_id.items()}
code_counts = Counter(id_to_code[e[-1]] for e in events)

RUN_TASK = {
    1: "Baseline, eyes open",
    2: "Baseline, eyes closed",
    3: "Motor execution: left vs right hand",
    4: "Motor imagery: left vs right hand",
    5: "Motor execution: hands vs feet",
    6: "Motor imagery: hands vs feet",
    7: "Motor execution: left vs right hand",
    8: "Motor imagery: left vs right hand",
    9: "Motor execution: hands vs feet",
    10: "Motor imagery: hands vs feet",
    11: "Motor execution: left vs right hand",
    12: "Motor imagery: left vs right hand",
    13: "Motor execution: hands vs feet",
    14: "Motor imagery: hands vs feet",
}

if RUN in {3, 4, 7, 8, 11, 12}:
    semantic_map = {"T0": "rest", "T1": "left fist", "T2": "right fist"}
elif RUN in {5, 6, 9, 10, 13, 14}:
    semantic_map = {"T0": "rest", "T1": "both fists", "T2": "both feet"}
else:
    semantic_map = {"T0": "rest"}

semantic_counts = {semantic_map.get(code, code): count for code, count in code_counts.items()}

print("Run:", RUN, "->", RUN_TASK.get(RUN, "Unknown run"))
print("event_id from MNE:", event_id)
print("Counts by raw code:", dict(code_counts))
print("Counts by semantic label:", semantic_counts)
print("First 5 events [sample, 0, id]:")
print(events[:5])


Run: 6 -> Motor imagery: hands vs feet
event_id from MNE: {np.str_('T0'): 1, np.str_('T1'): 2, np.str_('T2'): 3}
Counts by raw code: {np.str_('T0'): 15, np.str_('T2'): 8, np.str_('T1'): 7}
Counts by semantic label: {'rest': 15, 'both feet': 8, 'both fists': 7}
First 5 events [sample, 0, id]:
[[   0    0    1]
 [ 672    0    3]
 [1328    0    1]
 [2000    0    2]
 [2656    0    1]]


## Option 1: Feasibility test (Executed vs Imagined) with CSP + LDA

This section evaluates whether executed and imagined movement can be separated for the **same task family**:
- left/right fist: run pairs (3,4), (7,8), (11,12)
- both fists/feet: run pairs (5,6), (9,10), (13,14)

Labels are binary: `0 = executed`, `1 = imagined`.


In [49]:
# Install once if needed:
# %pip install scikit-learn

import time
import numpy as np
import mne
from mne.datasets import eegbci

try:
    from mne.decoding import CSP
    from sklearn.discriminant_analysis import LinearDiscriminantAnalysis
    from sklearn.metrics import balanced_accuracy_score
    from sklearn.pipeline import Pipeline
except ModuleNotFoundError as exc:
    raise ModuleNotFoundError(
        "scikit-learn is required for Option 1. Run: %pip install scikit-learn"
    ) from exc

# Keep logs compact during repeated CV fits
mne.set_log_level("WARNING")

# Evaluation settings
SUBJECTS = list(range(1, 110))  # many subjects
TASK_FAMILIES = ["left_right", "hands_feet"]
TMIN, TMAX = 0.5, 2.5  # seconds after cue
L_FREQ, H_FREQ = 8.0, 30.0
N_COMPONENTS = 6
PROGRESS_EVERY = 10  # print progress every N subjects

RUN_PAIR_FAMILIES = {
    "left_right": [(3, 4), (7, 8), (11, 12)],
    "hands_feet": [(5, 6), (9, 10), (13, 14)],
}

def load_task_epochs(subject, run, data_root, tmin, tmax, l_freq, h_freq):
    edf_path = data_root / f"S{subject:03d}" / f"S{subject:03d}R{run:02d}.edf"
    if not edf_path.exists():
        return None

    raw = mne.io.read_raw_edf(edf_path, infer_types=True, preload=True, verbose="ERROR")
    eegbci.standardize(raw)
    raw.set_montage(mne.channels.make_standard_montage("standard_1005"), on_missing="ignore")
    raw.pick("eeg")
    raw.filter(l_freq=l_freq, h_freq=h_freq, verbose="ERROR")

    events, event_id = mne.events_from_annotations(
        raw, event_id={"T1": 1, "T2": 2}, verbose="ERROR"
    )
    if len(events) == 0:
        return None

    epochs = mne.Epochs(
        raw,
        events,
        event_id=event_id,
        tmin=tmin,
        tmax=tmax,
        baseline=None,
        preload=True,
        verbose="ERROR",
    )
    return epochs

def build_exec_vs_imag_dataset(subject, task_family):
    pairs = RUN_PAIR_FAMILIES[task_family]
    X_all = []
    y_all = []
    groups_all = []

    for run_exec, run_imag in pairs:
        ep_exec = load_task_epochs(subject, run_exec, data_root, TMIN, TMAX, L_FREQ, H_FREQ)
        ep_imag = load_task_epochs(subject, run_imag, data_root, TMIN, TMAX, L_FREQ, H_FREQ)

        if ep_exec is None or ep_imag is None:
            continue

        X_exec = ep_exec.get_data(copy=False)
        X_imag = ep_imag.get_data(copy=False)

        X_all.append(X_exec)
        y_all.append(np.zeros(len(X_exec), dtype=int))
        groups_all.append(np.full(len(X_exec), run_exec, dtype=int))

        X_all.append(X_imag)
        y_all.append(np.ones(len(X_imag), dtype=int))
        groups_all.append(np.full(len(X_imag), run_imag, dtype=int))

    if not X_all:
        return None, None, None

    X = np.concatenate(X_all, axis=0)
    y = np.concatenate(y_all, axis=0)
    groups = np.concatenate(groups_all, axis=0)
    return X, y, groups

def make_leave_runpair_out_splits(groups, task_family):
    """Create grouped folds: each test fold holds out one exec+imag run pair."""
    pairs = RUN_PAIR_FAMILIES[task_family]
    unique_groups = set(np.unique(groups).tolist())
    splits = []

    for run_exec, run_imag in pairs:
        if run_exec not in unique_groups or run_imag not in unique_groups:
            continue

        test_mask = np.isin(groups, [run_exec, run_imag])
        train_idx = np.where(~test_mask)[0]
        test_idx = np.where(test_mask)[0]

        if len(train_idx) == 0 or len(test_idx) == 0:
            continue

        splits.append((train_idx, test_idx))

    return splits

def evaluate_subject_grouped(subject, task_family):
    X, y, groups = build_exec_vs_imag_dataset(subject, task_family)
    if X is None:
        return None

    splits = make_leave_runpair_out_splits(groups, task_family)
    if len(splits) < 2:
        return None

    clf = Pipeline([
        ("csp", CSP(n_components=N_COMPONENTS, reg="ledoit_wolf", log=True, norm_trace=False)),
        ("lda", LinearDiscriminantAnalysis(solver="lsqr", shrinkage="auto")),
    ])

    fold_scores = []
    for train_idx, test_idx in splits:
        X_train, y_train = X[train_idx], y[train_idx]
        X_test, y_test = X[test_idx], y[test_idx]

        clf.fit(X_train, y_train)
        y_pred = clf.predict(X_test)
        score = balanced_accuracy_score(y_test, y_pred)
        fold_scores.append(score)

    fold_scores = np.array(fold_scores, dtype=float)
    return {
        "subject": subject,
        "n_trials": int(len(y)),
        "n_exec": int((y == 0).sum()),
        "n_imag": int((y == 1).sum()),
        "n_folds": int(len(fold_scores)),
        "mean_bal_acc": float(np.mean(fold_scores)),
        "std_bal_acc": float(np.std(fold_scores)),
        "scores": fold_scores,
    }



In [50]:
all_family_results = {}

for task_family in TASK_FAMILIES:
    print(f"\n=== Evaluating task family: {task_family} ===")
    family_start = time.perf_counter()
    results = []

    total = len(SUBJECTS)
    for i, sub in enumerate(SUBJECTS, start=1):
        res = evaluate_subject_grouped(sub, task_family)
        if res is not None:
            results.append(res)

        if i % PROGRESS_EVERY == 0 or i == total:
            elapsed = time.perf_counter() - family_start
            print(
                f"[{task_family}] processed {i}/{total} subjects | "
                f"valid={len(results)} | elapsed={elapsed:.1f}s"
            )

    all_family_results[task_family] = results

# Detailed per-subject output (optional):
# for task_family, results in all_family_results.items():
#     for r in results:
#         print(
#             f"{task_family} S{r['subject']:03d} | folds={r['n_folds']} | "
#             f"balanced_acc={r['mean_bal_acc']:.3f} +/- {r['std_bal_acc']:.3f}"
#         )

print("\n=== Side-by-side summary ===")
print("family      | subjects | mean_bal_acc | std_across_subjects")
print("------------|----------|--------------|--------------------")

for task_family in TASK_FAMILIES:
    results = all_family_results.get(task_family, [])
    if not results:
        print(f"{task_family:11} | {0:8d} | {'n/a':>12} | {'n/a':>18}")
        continue

    subject_means = np.array([r['mean_bal_acc'] for r in results], dtype=float)
    print(
        f"{task_family:11} | {len(results):8d} | "
        f"{subject_means.mean():12.3f} | {subject_means.std():18.3f}"
    )

print("\nChance level is 0.500 for this binary task.")




=== Evaluating task family: left_right ===
[left_right] processed 10/109 subjects | valid=10 | elapsed=15.1s
[left_right] processed 20/109 subjects | valid=20 | elapsed=30.6s
[left_right] processed 30/109 subjects | valid=30 | elapsed=45.1s
[left_right] processed 40/109 subjects | valid=40 | elapsed=60.0s
[left_right] processed 50/109 subjects | valid=50 | elapsed=78.3s
[left_right] processed 60/109 subjects | valid=60 | elapsed=96.9s
[left_right] processed 70/109 subjects | valid=70 | elapsed=114.2s
[left_right] processed 80/109 subjects | valid=80 | elapsed=129.1s
[left_right] processed 90/109 subjects | valid=90 | elapsed=144.0s
[left_right] processed 100/109 subjects | valid=100 | elapsed=161.8s
[left_right] processed 109/109 subjects | valid=109 | elapsed=178.4s

=== Evaluating task family: hands_feet ===
[hands_feet] processed 10/109 subjects | valid=10 | elapsed=16.5s
[hands_feet] processed 20/109 subjects | valid=20 | elapsed=33.8s
[hands_feet] processed 30/109 subjects | vali

## Option 4: Compare all EEG channels vs sensorimotor subset

This cell runs the same grouped-CV evaluation with two channel modes:
- `all`: all EEG channels
- `sensorimotor`: channels around FC/C/CP (motor cortex neighborhood)

Use this to test whether restricting channels improves execution-vs-imagery classification.


In [51]:
# Channel subset comparison (uses same grouped run-pair CV idea)

# You can change these if needed:
COMPARE_TASK_FAMILIES = TASK_FAMILIES
COMPARE_SUBJECTS = SUBJECTS
COMPARE_PROGRESS_EVERY = 10

SENSORIMOTOR_CHANNELS = [
    "FC5", "FC3", "FC1", "FCz", "FC2", "FC4", "FC6",
    "C5", "C3", "C1", "Cz", "C2", "C4", "C6",
    "CP5", "CP3", "CP1", "CPz", "CP2", "CP4", "CP6",
]


def load_task_epochs_channel_mode(subject, run, data_root, tmin, tmax, l_freq, h_freq, channel_mode="all"):
    edf_path = data_root / f"S{subject:03d}" / f"S{subject:03d}R{run:02d}.edf"
    if not edf_path.exists():
        return None

    raw = mne.io.read_raw_edf(edf_path, infer_types=True, preload=True, verbose="ERROR")
    eegbci.standardize(raw)
    raw.set_montage(mne.channels.make_standard_montage("standard_1005"), on_missing="ignore")
    raw.pick("eeg")

    if channel_mode == "sensorimotor":
        picks = [ch for ch in SENSORIMOTOR_CHANNELS if ch in raw.ch_names]
        if len(picks) < 8:
            return None
        raw.pick(picks)

    raw.filter(l_freq=l_freq, h_freq=h_freq, verbose="ERROR")

    events, event_id = mne.events_from_annotations(
        raw, event_id={"T1": 1, "T2": 2}, verbose="ERROR"
    )
    if len(events) == 0:
        return None

    epochs = mne.Epochs(
        raw,
        events,
        event_id=event_id,
        tmin=tmin,
        tmax=tmax,
        baseline=None,
        preload=True,
        verbose="ERROR",
    )
    return epochs


def build_exec_vs_imag_dataset_channel_mode(subject, task_family, channel_mode="all"):
    pairs = RUN_PAIR_FAMILIES[task_family]
    X_all = []
    y_all = []
    groups_all = []

    for run_exec, run_imag in pairs:
        ep_exec = load_task_epochs_channel_mode(
            subject, run_exec, data_root, TMIN, TMAX, L_FREQ, H_FREQ, channel_mode=channel_mode
        )
        ep_imag = load_task_epochs_channel_mode(
            subject, run_imag, data_root, TMIN, TMAX, L_FREQ, H_FREQ, channel_mode=channel_mode
        )

        if ep_exec is None or ep_imag is None:
            continue

        X_exec = ep_exec.get_data(copy=False)
        X_imag = ep_imag.get_data(copy=False)

        X_all.append(X_exec)
        y_all.append(np.zeros(len(X_exec), dtype=int))
        groups_all.append(np.full(len(X_exec), run_exec, dtype=int))

        X_all.append(X_imag)
        y_all.append(np.ones(len(X_imag), dtype=int))
        groups_all.append(np.full(len(X_imag), run_imag, dtype=int))

    if not X_all:
        return None, None, None

    X = np.concatenate(X_all, axis=0)
    y = np.concatenate(y_all, axis=0)
    groups = np.concatenate(groups_all, axis=0)
    return X, y, groups


def evaluate_subject_grouped_channel_mode(subject, task_family, channel_mode="all"):
    X, y, groups = build_exec_vs_imag_dataset_channel_mode(subject, task_family, channel_mode=channel_mode)
    if X is None:
        return None

    splits = make_leave_runpair_out_splits(groups, task_family)
    if len(splits) < 2:
        return None

    clf = Pipeline([
        ("csp", CSP(n_components=N_COMPONENTS, reg="ledoit_wolf", log=True, norm_trace=False)),
        ("lda", LinearDiscriminantAnalysis(solver="lsqr", shrinkage="auto")),
    ])

    fold_scores = []
    for train_idx, test_idx in splits:
        X_train, y_train = X[train_idx], y[train_idx]
        X_test, y_test = X[test_idx], y[test_idx]

        clf.fit(X_train, y_train)
        y_pred = clf.predict(X_test)
        score = balanced_accuracy_score(y_test, y_pred)
        fold_scores.append(score)

    fold_scores = np.array(fold_scores, dtype=float)
    return {
        "subject": subject,
        "n_trials": int(len(y)),
        "n_channels": int(X.shape[1]),
        "mean_bal_acc": float(np.mean(fold_scores)),
        "std_bal_acc": float(np.std(fold_scores)),
    }


compare_results = {mode: {} for mode in ["all", "sensorimotor"]}

for mode in ["all", "sensorimotor"]:
    print(f"\n=== Channel mode: {mode} ===")
    mode_start = time.perf_counter()

    for task_family in COMPARE_TASK_FAMILIES:
        results = []
        total = len(COMPARE_SUBJECTS)

        for i, sub in enumerate(COMPARE_SUBJECTS, start=1):
            res = evaluate_subject_grouped_channel_mode(sub, task_family, channel_mode=mode)
            if res is not None:
                results.append(res)

            if i % COMPARE_PROGRESS_EVERY == 0 or i == total:
                elapsed = time.perf_counter() - mode_start
                print(
                    f"[{mode} | {task_family}] {i}/{total} subjects | "
                    f"valid={len(results)} | elapsed={elapsed:.1f}s"
                )

        compare_results[mode][task_family] = results


print("\n=== Side-by-side channel comparison ===")
print("mode         | family      | subjects | mean_bal_acc | std_across_subjects")
print("-------------|-------------|----------|--------------|--------------------")

for mode in ["all", "sensorimotor"]:
    for task_family in COMPARE_TASK_FAMILIES:
        results = compare_results[mode][task_family]
        if not results:
            print(f"{mode:12} | {task_family:11} | {0:8d} | {'n/a':>12} | {'n/a':>18}")
            continue

        subject_means = np.array([r["mean_bal_acc"] for r in results], dtype=float)
        print(
            f"{mode:12} | {task_family:11} | {len(results):8d} | "
            f"{subject_means.mean():12.3f} | {subject_means.std():18.3f}"
        )




=== Channel mode: all ===
[all | left_right] 10/109 subjects | valid=10 | elapsed=15.5s
[all | left_right] 20/109 subjects | valid=20 | elapsed=31.2s
[all | left_right] 30/109 subjects | valid=30 | elapsed=48.1s
[all | left_right] 40/109 subjects | valid=40 | elapsed=66.9s
[all | left_right] 50/109 subjects | valid=50 | elapsed=84.7s
[all | left_right] 60/109 subjects | valid=60 | elapsed=105.3s
[all | left_right] 70/109 subjects | valid=70 | elapsed=123.4s
[all | left_right] 80/109 subjects | valid=80 | elapsed=139.5s
[all | left_right] 90/109 subjects | valid=90 | elapsed=157.1s
[all | left_right] 100/109 subjects | valid=100 | elapsed=173.1s
[all | left_right] 109/109 subjects | valid=109 | elapsed=189.2s
[all | hands_feet] 10/109 subjects | valid=10 | elapsed=207.8s
[all | hands_feet] 20/109 subjects | valid=20 | elapsed=223.4s
[all | hands_feet] 30/109 subjects | valid=30 | elapsed=239.7s
[all | hands_feet] 40/109 subjects | valid=40 | elapsed=257.8s
[all | hands_feet] 50/109 sub

## Accuracy Tuning Grid (Top 3 Improvements)

This section benchmarks combinations of:
- epoch window (`TMIN`, `TMAX`)
- frequency band (`L_FREQ`, `H_FREQ`)
- CSP components + classifier
- optional lightweight artifact rejection

It uses the same grouped run-pair CV (no run leakage) and prints progress.


In [53]:
# Grid search for top 3 improvements

try:
    import pandas as pd
except ModuleNotFoundError:
    pd = None

from sklearn.svm import SVC
from sklearn.linear_model import LogisticRegression

# ---------------------------
# 1) What to sweep
# ---------------------------
GRID_TASK_FAMILY = "left_right"   # "left_right" or "hands_feet"
GRID_SUBJECTS = list(range(1, 31)) # start small for speed, then expand to 1..109
GRID_PROGRESS_EVERY = 5

GRID_WINDOWS = [
    (0.0, 2.0),
    (0.5, 2.5),
    (1.0, 3.0),
]

GRID_BANDS = [
    (6.0, 30.0),
    (8.0, 30.0),
    (8.0, 35.0),
]

GRID_N_COMPONENTS = [4, 6, 8, 10]
GRID_CLASSIFIERS = ["lda", "svm_linear"]
GRID_ARTIFACT_REJECTION = [False, True]

# Lightweight rejection threshold (volts). 150 uV is common for rough cleaning.
GRID_REJECT_EEG_UV = 150.0

# ---------------------------
# 2) Helpers
# ---------------------------

def _make_classifier(name):
    if name == "lda":
        return LinearDiscriminantAnalysis(solver="lsqr", shrinkage="auto")
    if name == "svm_linear":
        return SVC(kernel="linear", C=1.0)
    if name == "logreg":
        return LogisticRegression(max_iter=1000, solver="liblinear")
    raise ValueError(f"Unknown classifier: {name}")


def _load_epochs_cached(subject, run, tmin, tmax, l_freq, h_freq, artifact_reject, cache):
    key = (subject, run, tmin, tmax, l_freq, h_freq, artifact_reject)
    if key in cache:
        return cache[key]

    edf_path = data_root / f"S{subject:03d}" / f"S{subject:03d}R{run:02d}.edf"
    if not edf_path.exists():
        cache[key] = None
        return None

    raw = mne.io.read_raw_edf(edf_path, infer_types=True, preload=True, verbose="ERROR")
    eegbci.standardize(raw)
    raw.set_montage(mne.channels.make_standard_montage("standard_1005"), on_missing="ignore")
    raw.pick("eeg")
    raw.filter(l_freq=l_freq, h_freq=h_freq, verbose="ERROR")

    events, event_id = mne.events_from_annotations(raw, event_id={"T1": 1, "T2": 2}, verbose="ERROR")
    if len(events) == 0:
        cache[key] = None
        return None

    reject = None
    if artifact_reject:
        reject = {"eeg": GRID_REJECT_EEG_UV * 1e-6}

    epochs = mne.Epochs(
        raw,
        events,
        event_id=event_id,
        tmin=tmin,
        tmax=tmax,
        baseline=None,
        reject=reject,
        preload=True,
        verbose="ERROR",
    )

    if len(epochs) == 0:
        cache[key] = None
        return None

    cache[key] = epochs
    return epochs


def _build_dataset_for_subject(subject, task_family, tmin, tmax, l_freq, h_freq, artifact_reject, cache):
    pairs = RUN_PAIR_FAMILIES[task_family]
    X_all, y_all, groups_all = [], [], []

    for run_exec, run_imag in pairs:
        ep_exec = _load_epochs_cached(subject, run_exec, tmin, tmax, l_freq, h_freq, artifact_reject, cache)
        ep_imag = _load_epochs_cached(subject, run_imag, tmin, tmax, l_freq, h_freq, artifact_reject, cache)

        if ep_exec is None or ep_imag is None:
            continue

        X_exec = ep_exec.get_data(copy=False)
        X_imag = ep_imag.get_data(copy=False)

        X_all.append(X_exec)
        y_all.append(np.zeros(len(X_exec), dtype=int))
        groups_all.append(np.full(len(X_exec), run_exec, dtype=int))

        X_all.append(X_imag)
        y_all.append(np.ones(len(X_imag), dtype=int))
        groups_all.append(np.full(len(X_imag), run_imag, dtype=int))

    if not X_all:
        return None, None, None

    X = np.concatenate(X_all, axis=0)
    y = np.concatenate(y_all, axis=0)
    groups = np.concatenate(groups_all, axis=0)
    return X, y, groups


def _evaluate_subject_cfg(subject, task_family, tmin, tmax, l_freq, h_freq, n_components, clf_name, artifact_reject, cache):
    X, y, groups = _build_dataset_for_subject(subject, task_family, tmin, tmax, l_freq, h_freq, artifact_reject, cache)
    if X is None:
        return None

    splits = make_leave_runpair_out_splits(groups, task_family)
    if len(splits) < 2:
        return None

    fold_scores = []
    for train_idx, test_idx in splits:
        X_train, y_train = X[train_idx], y[train_idx]
        X_test, y_test = X[test_idx], y[test_idx]

        # Guard against tiny/degenerate folds after artifact rejection.
        train_classes = np.unique(y_train)
        test_classes = np.unique(y_test)
        if len(train_classes) < 2 or len(test_classes) < 2:
            continue
        if len(y_train) <= len(train_classes):
            continue

        csp = CSP(n_components=n_components, reg="ledoit_wolf", log=True, norm_trace=False)
        clf = _make_classifier(clf_name)

        try:
            X_train_csp = csp.fit_transform(X_train, y_train)
            X_test_csp = csp.transform(X_test)
            clf.fit(X_train_csp, y_train)
            y_pred = clf.predict(X_test_csp)
            fold_scores.append(balanced_accuracy_score(y_test, y_pred))
        except ValueError:
            # Skip folds where model assumptions break (e.g., too few samples).
            continue

    if not fold_scores:
        return None

    fold_scores = np.asarray(fold_scores, dtype=float)
    return float(np.mean(fold_scores))


# ---------------------------
# 3) Run the grid
# ---------------------------

grid_cache = {}
grid_rows = []

grid_configs = []
for tmin, tmax in GRID_WINDOWS:
    for l_freq, h_freq in GRID_BANDS:
        for n_components in GRID_N_COMPONENTS:
            for clf_name in GRID_CLASSIFIERS:
                for artifact_reject in GRID_ARTIFACT_REJECTION:
                    grid_configs.append((tmin, tmax, l_freq, h_freq, n_components, clf_name, artifact_reject))

start_all = time.perf_counter()
print(f"Total configs: {len(grid_configs)} | subjects per config: {len(GRID_SUBJECTS)}")

for cfg_idx, (tmin, tmax, l_freq, h_freq, n_components, clf_name, artifact_reject) in enumerate(grid_configs, start=1):
    cfg_start = time.perf_counter()
    subject_scores = []

    print(
        f"\n[Config {cfg_idx}/{len(grid_configs)}] "
        f"win=({tmin},{tmax}) band=({l_freq},{h_freq}) "
        f"csp={n_components} clf={clf_name} reject={artifact_reject}"
    )

    for i, sub in enumerate(GRID_SUBJECTS, start=1):
        score = _evaluate_subject_cfg(
            subject=sub,
            task_family=GRID_TASK_FAMILY,
            tmin=tmin,
            tmax=tmax,
            l_freq=l_freq,
            h_freq=h_freq,
            n_components=n_components,
            clf_name=clf_name,
            artifact_reject=artifact_reject,
            cache=grid_cache,
        )

        if score is not None:
            subject_scores.append(score)

        if i % GRID_PROGRESS_EVERY == 0 or i == len(GRID_SUBJECTS):
            elapsed = time.perf_counter() - cfg_start
            print(f"  progress {i}/{len(GRID_SUBJECTS)} | valid={len(subject_scores)} | elapsed={elapsed:.1f}s")

    if subject_scores:
        arr = np.asarray(subject_scores, dtype=float)
        row = {
            "task_family": GRID_TASK_FAMILY,
            "subjects": int(len(arr)),
            "tmin": tmin,
            "tmax": tmax,
            "l_freq": l_freq,
            "h_freq": h_freq,
            "n_components": int(n_components),
            "classifier": clf_name,
            "artifact_reject": bool(artifact_reject),
            "mean_bal_acc": float(arr.mean()),
            "std_bal_acc": float(arr.std()),
        }
        grid_rows.append(row)

        print(
            f"  result mean={row['mean_bal_acc']:.3f} std={row['std_bal_acc']:.3f} "
            f"(n={row['subjects']})"
        )
    else:
        print("  no valid subjects for this config")

print(f"\nGrid finished in {time.perf_counter() - start_all:.1f}s")

# ---------------------------
# 4) Ranked summary
# ---------------------------

if not grid_rows:
    print("No results to summarize.")
else:
    if pd is not None:
        grid_df = pd.DataFrame(grid_rows).sort_values(["mean_bal_acc", "std_bal_acc"], ascending=[False, True])
        print("\nTop 10 configs:")
        display(grid_df.head(10))
    else:
        sorted_rows = sorted(grid_rows, key=lambda r: (-r["mean_bal_acc"], r["std_bal_acc"]))
        print("\nTop 10 configs:")
        for r in sorted_rows[:10]:
            print(r)




Total configs: 144 | subjects per config: 30

[Config 1/144] win=(0.0,2.0) band=(6.0,30.0) csp=4 clf=lda reject=False
  progress 5/30 | valid=5 | elapsed=5.8s
  progress 10/30 | valid=10 | elapsed=11.9s
  progress 15/30 | valid=15 | elapsed=19.2s
  progress 20/30 | valid=20 | elapsed=26.7s
  progress 25/30 | valid=25 | elapsed=34.5s
  progress 30/30 | valid=30 | elapsed=42.1s
  result mean=0.634 std=0.129 (n=30)

[Config 2/144] win=(0.0,2.0) band=(6.0,30.0) csp=4 clf=lda reject=True
  progress 5/30 | valid=2 | elapsed=5.0s
  progress 10/30 | valid=5 | elapsed=10.5s
  progress 15/30 | valid=8 | elapsed=16.1s
  progress 20/30 | valid=11 | elapsed=21.7s
  progress 25/30 | valid=13 | elapsed=26.9s
  progress 30/30 | valid=17 | elapsed=33.8s
  result mean=0.601 std=0.102 (n=17)

[Config 3/144] win=(0.0,2.0) band=(6.0,30.0) csp=4 clf=svm_linear reject=False
  progress 5/30 | valid=5 | elapsed=3.4s
  progress 10/30 | valid=10 | elapsed=6.6s
  progress 15/30 | valid=15 | elapsed=9.9s
  progres

C:\Users\Kades\AppData\Roaming\Python\Python314\site-packages\sklearn\covariance\_shrunk_covariance.py:349: UserWarning: Only one sample available. You may want to reshape your data array
  warnings.warn(
C:\Users\Kades\AppData\Roaming\Python\Python314\site-packages\sklearn\covariance\_empirical_covariance.py:100: UserWarning: Only one sample available. You may want to reshape your data array
  warnings.warn(


  progress 15/30 | valid=8 | elapsed=6.9s
  progress 20/30 | valid=11 | elapsed=9.8s
  progress 25/30 | valid=13 | elapsed=12.0s
  progress 30/30 | valid=17 | elapsed=14.7s
  result mean=0.637 std=0.167 (n=17)

[Config 83/144] win=(0.5,2.5) band=(8.0,35.0) csp=4 clf=svm_linear reject=False
  progress 5/30 | valid=5 | elapsed=1.7s
  progress 10/30 | valid=10 | elapsed=3.2s
  progress 15/30 | valid=15 | elapsed=4.8s
  progress 20/30 | valid=20 | elapsed=6.3s
  progress 25/30 | valid=25 | elapsed=7.8s
  progress 30/30 | valid=30 | elapsed=9.2s
  result mean=0.664 std=0.168 (n=30)

[Config 84/144] win=(0.5,2.5) band=(8.0,35.0) csp=4 clf=svm_linear reject=True
  progress 5/30 | valid=2 | elapsed=0.7s
  progress 10/30 | valid=5 | elapsed=1.6s
  progress 15/30 | valid=8 | elapsed=2.5s
  progress 20/30 | valid=11 | elapsed=3.5s
  progress 25/30 | valid=13 | elapsed=4.3s
  progress 30/30 | valid=17 | elapsed=5.7s
  result mean=0.629 std=0.156 (n=17)

[Config 85/144] win=(0.5,2.5) band=(8.0,35.0

C:\Users\Kades\AppData\Roaming\Python\Python314\site-packages\sklearn\covariance\_shrunk_covariance.py:349: UserWarning: Only one sample available. You may want to reshape your data array
  warnings.warn(
C:\Users\Kades\AppData\Roaming\Python\Python314\site-packages\sklearn\covariance\_empirical_covariance.py:100: UserWarning: Only one sample available. You may want to reshape your data array
  warnings.warn(


  progress 15/30 | valid=8 | elapsed=2.3s
  progress 20/30 | valid=11 | elapsed=3.3s
  progress 25/30 | valid=13 | elapsed=4.0s
  progress 30/30 | valid=17 | elapsed=5.4s
  result mean=0.639 std=0.168 (n=17)

[Config 87/144] win=(0.5,2.5) band=(8.0,35.0) csp=6 clf=svm_linear reject=False
  progress 5/30 | valid=5 | elapsed=2.1s
  progress 10/30 | valid=10 | elapsed=4.4s
  progress 15/30 | valid=15 | elapsed=6.6s
  progress 20/30 | valid=20 | elapsed=8.9s
  progress 25/30 | valid=25 | elapsed=11.1s
  progress 30/30 | valid=30 | elapsed=13.4s
  result mean=0.660 std=0.160 (n=30)

[Config 88/144] win=(0.5,2.5) band=(8.0,35.0) csp=6 clf=svm_linear reject=True
  progress 5/30 | valid=2 | elapsed=0.7s
  progress 10/30 | valid=5 | elapsed=1.6s
  progress 15/30 | valid=8 | elapsed=2.5s
  progress 20/30 | valid=11 | elapsed=3.4s
  progress 25/30 | valid=13 | elapsed=4.1s
  progress 30/30 | valid=17 | elapsed=5.3s
  result mean=0.633 std=0.145 (n=17)

[Config 89/144] win=(0.5,2.5) band=(8.0,35.0

...

C:\Users\Kades\AppData\Roaming\Python\Python314\site-packages\sklearn\covariance\_shrunk_covariance.py:349: UserWarning: Only one sample available. You may want to reshape your data array
  warnings.warn(
C:\Users\Kades\AppData\Roaming\Python\Python314\site-packages\sklearn\covariance\_empirical_covariance.py:100: UserWarning: Only one sample available. You may want to reshape your data array
  warnings.warn(


  progress 15/30 | valid=8 | elapsed=2.3s
  progress 20/30 | valid=11 | elapsed=3.2s
  progress 25/30 | valid=13 | elapsed=3.9s
  progress 30/30 | valid=17 | elapsed=5.2s
  result mean=0.637 std=0.146 (n=17)

[Config 139/144] win=(1.0,3.0) band=(8.0,35.0) csp=8 clf=svm_linear reject=False
  progress 5/30 | valid=5 | elapsed=2.1s
  progress 10/30 | valid=10 | elapsed=4.2s
  progress 15/30 | valid=15 | elapsed=6.2s
  progress 20/30 | valid=20 | elapsed=8.3s
  progress 25/30 | valid=25 | elapsed=10.3s
  progress 30/30 | valid=30 | elapsed=12.3s
  result mean=0.651 std=0.147 (n=30)

[Config 140/144] win=(1.0,3.0) band=(8.0,35.0) csp=8 clf=svm_linear reject=True
  progress 5/30 | valid=2 | elapsed=0.6s
  progress 10/30 | valid=5 | elapsed=1.4s
  progress 15/30 | valid=8 | elapsed=2.2s
  progress 20/30 | valid=11 | elapsed=3.2s
  progress 25/30 | valid=13 | elapsed=3.8s
  progress 30/30 | valid=17 | elapsed=5.2s
  result mean=0.622 std=0.130 (n=17)

[Config 141/144] win=(1.0,3.0) band=(8.0,3

C:\Users\Kades\AppData\Roaming\Python\Python314\site-packages\sklearn\covariance\_shrunk_covariance.py:349: UserWarning: Only one sample available. You may want to reshape your data array
  warnings.warn(
C:\Users\Kades\AppData\Roaming\Python\Python314\site-packages\sklearn\covariance\_empirical_covariance.py:100: UserWarning: Only one sample available. You may want to reshape your data array
  warnings.warn(


  progress 15/30 | valid=8 | elapsed=2.3s
  progress 20/30 | valid=11 | elapsed=3.3s
  progress 25/30 | valid=13 | elapsed=4.0s
  progress 30/30 | valid=17 | elapsed=5.4s
  result mean=0.640 std=0.137 (n=17)

[Config 143/144] win=(1.0,3.0) band=(8.0,35.0) csp=10 clf=svm_linear reject=False
  progress 5/30 | valid=5 | elapsed=2.1s
  progress 10/30 | valid=10 | elapsed=4.2s
  progress 15/30 | valid=15 | elapsed=6.3s
  progress 20/30 | valid=20 | elapsed=8.3s
  progress 25/30 | valid=25 | elapsed=10.3s
  progress 30/30 | valid=30 | elapsed=12.2s
  result mean=0.650 std=0.142 (n=30)

[Config 144/144] win=(1.0,3.0) band=(8.0,35.0) csp=10 clf=svm_linear reject=True
  progress 5/30 | valid=2 | elapsed=0.6s
  progress 10/30 | valid=5 | elapsed=1.4s
  progress 15/30 | valid=8 | elapsed=2.3s
  progress 20/30 | valid=11 | elapsed=3.3s
  progress 25/30 | valid=13 | elapsed=3.9s
  progress 30/30 | valid=17 | elapsed=5.3s
  result mean=0.623 std=0.129 (n=17)

Grid finished in 1755.6s

Top 10 configs

,task_family,subjects,tmin,tmax,l_freq,h_freq,n_components,classifier,artifact_reject,mean_bal_acc,std_bal_acc
100,left_right,30,1.0,3.0,6.0,30.0,6,lda,False,0.687407,0.133325
96,left_right,30,1.0,3.0,6.0,30.0,4,lda,False,0.686296,0.134491
75,left_right,17,0.5,2.5,8.0,30.0,8,svm_linear,True,0.684497,0.138762
128,left_right,30,1.0,3.0,8.0,35.0,4,lda,False,0.680741,0.147432
60,left_right,30,0.5,2.5,6.0,30.0,10,lda,False,0.678519,0.137165
48,left_right,30,0.5,2.5,6.0,30.0,4,lda,False,0.678519,0.142896
112,left_right,30,1.0,3.0,8.0,30.0,4,lda,False,0.678148,0.146523
104,left_right,30,1.0,3.0,6.0,30.0,8,lda,False,0.677778,0.125051
62,left_right,30,0.5,2.5,6.0,30.0,10,svm_linear,False,0.677407,0.136007
102,left_right,30,1.0,3.0,6.0,30.0,6,svm_linear,False,0.677037,0.132713


## Full Test With Winning Settings

Runs grouped run-pair CV across all subjects using the winning configuration from the tuning grid.


In [55]:
# Full evaluation with winning settings from the tuning grid (all task families)

FULL_TASK_FAMILIES = ["left_right", "hands_feet"]
FULL_SUBJECTS = list(range(1, 110))
FULL_PROGRESS_EVERY = 10

# Winning settings from tuning (tuned on left_right)
FULL_TMIN, FULL_TMAX = 1.0, 3.0
FULL_L_FREQ, FULL_H_FREQ = 6.0, 30.0
FULL_N_COMPONENTS = 6
FULL_CLASSIFIER = "lda"
FULL_ARTIFACT_REJECT = False
FULL_REJECT_EEG_UV = 150.0


def full_make_classifier(name):
    if name == "lda":
        return LinearDiscriminantAnalysis(solver="lsqr", shrinkage="auto")
    if name == "svm_linear":
        return SVC(kernel="linear", C=1.0)
    if name == "logreg":
        return LogisticRegression(max_iter=1000, solver="liblinear")
    raise ValueError(f"Unknown classifier: {name}")


def full_load_epochs(subject, run, tmin, tmax, l_freq, h_freq, artifact_reject):
    edf_path = data_root / f"S{subject:03d}" / f"S{subject:03d}R{run:02d}.edf"
    if not edf_path.exists():
        return None

    raw = mne.io.read_raw_edf(edf_path, infer_types=True, preload=True, verbose="ERROR")
    eegbci.standardize(raw)
    raw.set_montage(mne.channels.make_standard_montage("standard_1005"), on_missing="ignore")
    raw.pick("eeg")
    raw.filter(l_freq=l_freq, h_freq=h_freq, verbose="ERROR")

    events, event_id = mne.events_from_annotations(raw, event_id={"T1": 1, "T2": 2}, verbose="ERROR")
    if len(events) == 0:
        return None

    reject = {"eeg": FULL_REJECT_EEG_UV * 1e-6} if artifact_reject else None

    epochs = mne.Epochs(
        raw,
        events,
        event_id=event_id,
        tmin=tmin,
        tmax=tmax,
        baseline=None,
        reject=reject,
        preload=True,
        verbose="ERROR",
    )
    if len(epochs) == 0:
        return None

    return epochs


def full_build_dataset(subject, task_family):
    pairs = RUN_PAIR_FAMILIES[task_family]
    X_all, y_all, groups_all = [], [], []

    for run_exec, run_imag in pairs:
        ep_exec = full_load_epochs(subject, run_exec, FULL_TMIN, FULL_TMAX, FULL_L_FREQ, FULL_H_FREQ, FULL_ARTIFACT_REJECT)
        ep_imag = full_load_epochs(subject, run_imag, FULL_TMIN, FULL_TMAX, FULL_L_FREQ, FULL_H_FREQ, FULL_ARTIFACT_REJECT)

        if ep_exec is None or ep_imag is None:
            continue

        X_exec = ep_exec.get_data(copy=False)
        X_imag = ep_imag.get_data(copy=False)

        X_all.append(X_exec)
        y_all.append(np.zeros(len(X_exec), dtype=int))
        groups_all.append(np.full(len(X_exec), run_exec, dtype=int))

        X_all.append(X_imag)
        y_all.append(np.ones(len(X_imag), dtype=int))
        groups_all.append(np.full(len(X_imag), run_imag, dtype=int))

    if not X_all:
        return None, None, None

    X = np.concatenate(X_all, axis=0)
    y = np.concatenate(y_all, axis=0)
    groups = np.concatenate(groups_all, axis=0)
    return X, y, groups


def full_evaluate_subject(subject, task_family):
    X, y, groups = full_build_dataset(subject, task_family)
    if X is None:
        return None

    splits = make_leave_runpair_out_splits(groups, task_family)
    if len(splits) < 2:
        return None

    fold_scores = []
    for train_idx, test_idx in splits:
        X_train, y_train = X[train_idx], y[train_idx]
        X_test, y_test = X[test_idx], y[test_idx]

        train_classes = np.unique(y_train)
        test_classes = np.unique(y_test)
        if len(train_classes) < 2 or len(test_classes) < 2:
            continue
        if len(y_train) <= len(train_classes):
            continue

        csp = CSP(n_components=FULL_N_COMPONENTS, reg="ledoit_wolf", log=True, norm_trace=False)
        clf = full_make_classifier(FULL_CLASSIFIER)

        try:
            X_train_csp = csp.fit_transform(X_train, y_train)
            X_test_csp = csp.transform(X_test)
            clf.fit(X_train_csp, y_train)
            y_pred = clf.predict(X_test_csp)
            fold_scores.append(balanced_accuracy_score(y_test, y_pred))
        except ValueError:
            continue

    if not fold_scores:
        return None

    fold_scores = np.asarray(fold_scores, dtype=float)
    return {
        "subject": subject,
        "n_trials": int(len(y)),
        "n_folds": int(len(fold_scores)),
        "mean_bal_acc": float(np.mean(fold_scores)),
        "std_bal_acc": float(np.std(fold_scores)),
    }


print("Running full grouped-CV test with winning settings:")
print(
    f"window=({FULL_TMIN},{FULL_TMAX}), band=({FULL_L_FREQ},{FULL_H_FREQ}), "
    f"csp={FULL_N_COMPONENTS}, clf={FULL_CLASSIFIER}, artifact_reject={FULL_ARTIFACT_REJECT}"
)

all_family_results = {}
for family in FULL_TASK_FAMILIES:
    print(f"\n=== Family: {family} ===")
    start = time.perf_counter()
    results = []

    total = len(FULL_SUBJECTS)
    for i, sub in enumerate(FULL_SUBJECTS, start=1):
        r = full_evaluate_subject(sub, family)
        if r is not None:
            results.append(r)

        if i % FULL_PROGRESS_EVERY == 0 or i == total:
            elapsed = time.perf_counter() - start
            print(f"[{family}] progress {i}/{total} | valid={len(results)} | elapsed={elapsed:.1f}s")

    all_family_results[family] = results

print("\nFull-test summary (all families)")
print("family      | subjects | mean_bal_acc | std_across_subjects")
print("------------|----------|--------------|--------------------")
for family in FULL_TASK_FAMILIES:
    rows = all_family_results.get(family, [])
    if not rows:
        print(f"{family:11} | {0:8d} | {'n/a':>12} | {'n/a':>18}")
        continue
    vals = np.array([r['mean_bal_acc'] for r in rows], dtype=float)
    print(f"{family:11} | {len(rows):8d} | {vals.mean():12.3f} | {vals.std():18.3f}")

print("chance level: 0.500")



Running full grouped-CV test with winning settings:
window=(1.0,3.0), band=(6.0,30.0), csp=6, clf=lda, artifact_reject=False

=== Family: left_right ===
[left_right] progress 10/109 | valid=10 | elapsed=6.8s
[left_right] progress 20/109 | valid=20 | elapsed=13.0s
[left_right] progress 30/109 | valid=30 | elapsed=19.2s
[left_right] progress 40/109 | valid=40 | elapsed=27.2s
[left_right] progress 50/109 | valid=50 | elapsed=34.6s
[left_right] progress 60/109 | valid=60 | elapsed=40.4s
[left_right] progress 70/109 | valid=70 | elapsed=46.2s
[left_right] progress 80/109 | valid=80 | elapsed=52.6s
[left_right] progress 90/109 | valid=90 | elapsed=58.5s
[left_right] progress 100/109 | valid=100 | elapsed=64.3s
[left_right] progress 109/109 | valid=109 | elapsed=69.6s

=== Family: hands_feet ===
[hands_feet] progress 10/109 | valid=10 | elapsed=6.4s
[hands_feet] progress 20/109 | valid=20 | elapsed=13.1s
[hands_feet] progress 30/109 | valid=30 | elapsed=20.1s
[hands_feet] progress 40/109 | va